In [1]:
import torch

from models.swin3d_model import Model
from losses.generalized_dice import GeneralizedDiceLoss
from configs.swin3d_small import TRAIN_CONFIG


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Model(
    num_classes=TRAIN_CONFIG["num_classes"],
    in_channels=4,
).to(device)

# Explicitly train all Swin3D layers.
for parameter in model.parameters():
    parameter.requires_grad = True

criterion = GeneralizedDiceLoss(
    num_classes=TRAIN_CONFIG["num_classes"],
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TRAIN_CONFIG["max_epochs"],
    eta_min=1e-7,
)

print(model)
print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Trainable parameters:", sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
))

/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


Model(
  (backbone): Swin3DUNet(
    (stem_layer): MinkConvBNRelu(
      (conv_layers): Sequential(
        (0): MinkowskiConvolution(in=4, out=48, kernel_size=[3, 3, 3], stride=[1, 1, 1], dilation=[1, 1, 1])
        (1): MinkowskiBatchNorm(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): MinkowskiReLU()
      )
    )
    (layers): ModuleList(
      (0): BasicLayer(
        window_size=5, depth=2, channel=48, num_heads=6, quant_size=4, cRSE=XYZ, fp16_mode=1
        (blocks): ModuleList(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=48, out_features=144, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=True)
              (proj): Linear(in_features=48, out_features=48, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=True)
              (softmax): Softmax(dim=-1)
            )
            (dr

In [2]:
import torch


def move_batch_to_device(batch, device):
    coords, features, offset, labels, paths = batch

    coords = coords.to(
        device,
        non_blocking=True,
    ).float()

    features = features.to(
        device,
        non_blocking=True,
    ).float()

    offset = offset.to(
        device,
        non_blocking=True,
    ).long()

    labels = labels.to(
        device,
        non_blocking=True,
    ).long()

    return coords, features, offset, labels


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_points = 0

    for batch in loader:
        coords, features, offset, labels = move_batch_to_device(
            batch,
            device,
        )

        optimizer.zero_grad(set_to_none=True)

        # Model expects:
        # (coords, features, offset)
        logits = model(
            (coords, features, offset)
        )

        # Some Swin3D implementations may return a tuple.
        if isinstance(logits, tuple):
            logits = logits[0]

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        predictions = logits.argmax(dim=1)

        number_of_points = labels.numel()

        total_loss += loss.item() * number_of_points
        total_correct += (
            predictions == labels
        ).sum().item()
        total_points += number_of_points

    average_loss = total_loss / max(total_points, 1)
    accuracy = total_correct / max(total_points, 1)

    return average_loss, accuracy


@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_points = 0

    for batch in loader:
        coords, features, offset, labels = move_batch_to_device(
            batch,
            device,
        )

        logits = model(
            (coords, features, offset)
        )

        if isinstance(logits, tuple):
            logits = logits[0]

        loss = criterion(logits, labels)

        predictions = logits.argmax(dim=1)

        number_of_points = labels.numel()

        total_loss += loss.item() * number_of_points
        total_correct += (
            predictions == labels
        ).sum().item()
        total_points += number_of_points

    average_loss = total_loss / max(total_points, 1)
    accuracy = total_correct / max(total_points, 1)

    return average_loss, accuracy

In [3]:
from datasets.belize_laz_dataset import create_dataloaders


train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=TRAIN_CONFIG["batch_size"],
    voxel_max=TRAIN_CONFIG["voxel_max"],
    num_workers=4,
    augmentation_ratio=TRAIN_CONFIG["augmentation_ratio"],
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

Training batches: 46
Validation batches: 10
Test batches: 10


In [4]:
from pathlib import Path

checkpoint_dir = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

In [5]:
coords, features, offset, labels, paths = next(
    iter(train_loader)
)

coords = coords.to(device)
features = features.to(device)
offset = offset.to(device)
labels = labels.to(device)

print("Model input channels:", coords.shape[1] + features.shape[1])

with torch.no_grad():
    logits = model(
        (coords, features, offset)
    )

if isinstance(logits, tuple):
    logits = logits[0]

print("Logits:", logits.shape)
print("Labels:", labels.shape)

Model input channels: 4


/work/archcomplab/ssaraf/envs/maya_env/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/work/archcomplab/ssaraf/software/Swin3D/Swin3D/modules/swin3d_layers.py:239: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:78.)
  offset = torch.cuda.IntTensor(offset)


Logits: torch.Size([520000, 3])
Labels: torch.Size([520000])


In [6]:
model.train()

coords, features, offset, labels, paths = next(iter(train_loader))

coords = coords.to(device, non_blocking=True).float()
features = features.to(device, non_blocking=True).float()
offset = offset.to(device, non_blocking=True).long()
labels = labels.to(device, non_blocking=True).long()

optimizer.zero_grad(set_to_none=True)

logits = model(
    (coords, features, offset)
)

if isinstance(logits, tuple):
    logits = logits[0]

loss = criterion(logits, labels)

loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0,
)

optimizer.step()

print("One training step completed.")
print("Loss:", loss.item())

One training step completed.
Loss: 0.9928767681121826


In [7]:
import torch


def move_batch_to_device(batch, device):
    coords, features, offset, labels, paths = batch

    coords = coords.to(
        device,
        non_blocking=True,
    ).float()

    features = features.to(
        device,
        non_blocking=True,
    ).float()

    offset = offset.to(
        device,
        non_blocking=True,
    ).long()

    labels = labels.to(
        device,
        non_blocking=True,
    ).long()

    return coords, features, offset, labels


def calculate_batch_accuracy(logits, labels):
    predictions = logits.argmax(dim=1)

    correct = (
        predictions == labels
    ).sum().item()

    total = labels.numel()

    return correct, total


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_points = 0

    for batch_index, batch in enumerate(loader):
        coords, features, offset, labels = (
            move_batch_to_device(
                batch,
                device,
            )
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            (coords, features, offset)
        )

        if isinstance(logits, tuple):
            logits = logits[0]

        if logits.shape[0] != labels.shape[0]:
            raise RuntimeError(
                "Logit and label point counts do not match: "
                f"logits={logits.shape}, labels={labels.shape}"
            )

        loss = criterion(
            logits,
            labels,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss detected in batch {batch_index}: "
                f"{loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        correct, number_of_points = (
            calculate_batch_accuracy(
                logits,
                labels,
            )
        )

        total_loss += (
            loss.item() * number_of_points
        )

        total_correct += correct
        total_points += number_of_points

        if (
            batch_index == 0
            or (batch_index + 1) % 10 == 0
        ):
            print(
                f"  Batch {batch_index + 1:04d}/"
                f"{len(loader):04d} | "
                f"Loss: {loss.item():.6f}"
            )

        del coords
        del features
        del offset
        del labels
        del logits
        del loss

    average_loss = (
        total_loss / max(total_points, 1)
    )

    accuracy = (
        total_correct / max(total_points, 1)
    )

    return average_loss, accuracy


@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_points = 0

    for batch in loader:
        coords, features, offset, labels = (
            move_batch_to_device(
                batch,
                device,
            )
        )

        logits = model(
            (coords, features, offset)
        )

        if isinstance(logits, tuple):
            logits = logits[0]

        if logits.shape[0] != labels.shape[0]:
            raise RuntimeError(
                "Logit and label point counts do not match: "
                f"logits={logits.shape}, labels={labels.shape}"
            )

        loss = criterion(
            logits,
            labels,
        )

        correct, number_of_points = (
            calculate_batch_accuracy(
                logits,
                labels,
            )
        )

        total_loss += (
            loss.item() * number_of_points
        )

        total_correct += correct
        total_points += number_of_points

    average_loss = (
        total_loss / max(total_points, 1)
    )

    accuracy = (
        total_correct / max(total_points, 1)
    )

    return average_loss, accuracy

In [8]:
from pathlib import Path

checkpoint_dir = Path(
    "/mnt/nrdstor/archcomplab/ssaraf/"
    "swin3d_checkpoints/"
    "generalized_dice"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

best_checkpoint_path = (
    checkpoint_dir /
    "swin3d_generalized_dice_best.pth"
)

last_checkpoint_path = (
    checkpoint_dir /
    "swin3d_generalized_dice_last.pth"
)

print("Best checkpoint:", best_checkpoint_path)
print("Last checkpoint:", last_checkpoint_path)

Best checkpoint: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/generalized_dice/swin3d_generalized_dice_best.pth
Last checkpoint: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/generalized_dice/swin3d_generalized_dice_last.pth


In [9]:
test_epochs = 2

for epoch in range(test_epochs):
    train_loss, train_accuracy = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    validation_loss, validation_accuracy = validate(
        model=model,
        loader=validation_loader,
        criterion=criterion,
        device=device,
    )

    scheduler.step()

    print(
        f"\nEpoch {epoch + 1}/{test_epochs}\n"
        f"Train loss: {train_loss:.6f}\n"
        f"Train accuracy: {train_accuracy:.4f}\n"
        f"Validation loss: {validation_loss:.6f}\n"
        f"Validation accuracy: {validation_accuracy:.4f}\n"
    )

  Batch 0001/0046 | Loss: 0.996757
  Batch 0010/0046 | Loss: 0.983771
  Batch 0020/0046 | Loss: 0.992293
  Batch 0030/0046 | Loss: 0.997818
  Batch 0040/0046 | Loss: 0.990531

Epoch 1/2
Train loss: 0.978892
Train accuracy: 0.4110
Validation loss: 0.973748
Validation accuracy: 0.5905

  Batch 0001/0046 | Loss: 0.990219
  Batch 0010/0046 | Loss: 0.984786
  Batch 0020/0046 | Loss: 0.987915
  Batch 0030/0046 | Loss: 0.984075
  Batch 0040/0046 | Loss: 0.990894

Epoch 2/2
Train loss: 0.976027
Train accuracy: 0.6283
Validation loss: 0.972519
Validation accuracy: 0.5925



In [10]:
from models.swin3d_model import Model
from losses.generalized_dice import GeneralizedDiceLoss


model = Model(
    num_classes=TRAIN_CONFIG["num_classes"],
    in_channels=4,
).to(device)

criterion = GeneralizedDiceLoss(
    num_classes=TRAIN_CONFIG["num_classes"],
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TRAIN_CONFIG["max_epochs"],
    eta_min=1e-7,
)

In [11]:
best_validation_loss = float("inf")
epochs_without_improvement = 0

max_epochs = TRAIN_CONFIG["max_epochs"]
patience = TRAIN_CONFIG["early_stopping_patience"]

training_history = {
    "train_loss": [],
    "train_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
    "learning_rate": [],
}


for epoch in range(max_epochs):
    print(
        f"\n{'=' * 60}\n"
        f"Epoch {epoch + 1}/{max_epochs}\n"
        f"{'=' * 60}"
    )

    train_loss, train_accuracy = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    validation_loss, validation_accuracy = validate(
        model=model,
        loader=validation_loader,
        criterion=criterion,
        device=device,
    )

    current_learning_rate = (
        optimizer.param_groups[0]["lr"]
    )

    training_history["train_loss"].append(
        train_loss
    )

    training_history["train_accuracy"].append(
        train_accuracy
    )

    training_history["validation_loss"].append(
        validation_loss
    )

    training_history["validation_accuracy"].append(
        validation_accuracy
    )

    training_history["learning_rate"].append(
        current_learning_rate
    )

    print(
        f"\nEpoch {epoch + 1:03d} results\n"
        f"Train loss:       {train_loss:.6f}\n"
        f"Train accuracy:   {train_accuracy:.4f}\n"
        f"Validation loss:  {validation_loss:.6f}\n"
        f"Validation acc:   {validation_accuracy:.4f}\n"
        f"Learning rate:    {current_learning_rate:.8f}"
    )

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
        "best_validation_loss": best_validation_loss,
        "training_history": training_history,
        "config": TRAIN_CONFIG,
    }

    torch.save(
        checkpoint,
        last_checkpoint_path,
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0

        checkpoint["best_validation_loss"] = (
            best_validation_loss
        )

        torch.save(
            checkpoint,
            best_checkpoint_path,
        )

        print(
            f"Saved new best model: "
            f"{best_checkpoint_path}"
        )

    else:
        epochs_without_improvement += 1

        print(
            "No validation improvement: "
            f"{epochs_without_improvement}/{patience}"
        )

    scheduler.step()

    if epochs_without_improvement >= patience:
        print(
            f"\nEarly stopping at epoch {epoch + 1}."
        )
        print(
            f"Best validation loss: "
            f"{best_validation_loss:.6f}"
        )
        break

    torch.cuda.empty_cache()


Epoch 1/100
  Batch 0001/0046 | Loss: 0.986741
  Batch 0010/0046 | Loss: 0.964052
  Batch 0020/0046 | Loss: 0.973755
  Batch 0030/0046 | Loss: 0.965926
  Batch 0040/0046 | Loss: 0.974947

Epoch 001 results
Train loss:       0.978018
Train accuracy:   0.4158
Validation loss:  0.973727
Validation acc:   0.4572
Learning rate:    0.00003000
Saved new best model: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/generalized_dice/swin3d_generalized_dice_best.pth

Epoch 2/100
  Batch 0001/0046 | Loss: 0.989670
  Batch 0010/0046 | Loss: 0.920693
  Batch 0020/0046 | Loss: 0.967704
  Batch 0030/0046 | Loss: 0.997106
  Batch 0040/0046 | Loss: 0.993429

Epoch 002 results
Train loss:       0.974569
Train accuracy:   0.5503
Validation loss:  0.969915
Validation acc:   0.6442
Learning rate:    0.00002999
Saved new best model: /mnt/nrdstor/archcomplab/ssaraf/swin3d_checkpoints/generalized_dice/swin3d_generalized_dice_best.pth

Epoch 3/100
  Batch 0001/0046 | Loss: 0.939782
  Batch 0010/0046 | Loss: 

norm, x,y: center of tile
      z  : median of z range of the tile.